# Reactive (token-conditional) steering on Qwen3-0.6B

Three-arm comparison at matched coef (`additive_normalized`):
- `always_on` — perturbation at every decode step.
- `reactive_detect` — gated by the binary `is_pivotal` probe at layer 14, hysteresis=2.
- `reactive_random` — Bernoulli mask matched to the empirical fire rate of `reactive_detect`.

Closes Issue #4 from `docs/issues.md`. The signed/two-probe cascade lives in a separate plan; this notebook is intentionally detection-only.


## 1. Install


In [ ]:
# Run once per runtime. Safe to skip if already installed.
%pip install -q "transformers>=4.44" datasets accelerate matplotlib seaborn scikit-learn torch numpy pandas


## 2. Imports + seed


In [ ]:
import json
import os
import random
import re
import sys
import time
from pathlib import Path

import numpy as np
import torch

# Notebook is also runnable from the repo root checkout, where probe_pipeline
# is importable. On Colab, clone the repo first (cell after this).
try:
    from probe_pipeline.steering_reactive import (
        ReactiveSteeringHook, ReactiveSteeringStats, build_random_fire_pattern,
    )
    from probe_pipeline.steering import _get_decoder_layer, make_hook
    print("[imports] probe_pipeline available locally")
except Exception:
    print("[imports] probe_pipeline NOT importable; clone the repo (next cell) and re-run")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


## 2b. Colab clone (no-op locally)


In [ ]:
# Colab-friendly clone (no-op locally). After this cell, restart-and-run the
# imports cell above so probe_pipeline.* resolves.
import os, subprocess, sys
REPO_URL = "https://github.com/stvngo/Pivotal-Token-Representation-Learning.git"
REPO_DIR = "Pivotal-Token-Representation-Learning"
if not os.path.isdir(REPO_DIR) and "/content" in os.getcwd():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL])
    sys.path.insert(0, os.path.abspath(REPO_DIR))
elif "probe_pipeline" not in sys.modules:
    here = Path.cwd()
    for parent in [here, *here.parents]:
        if (parent / "probe_pipeline").exists():
            sys.path.insert(0, str(parent))
            break
print("sys.path[0]:", sys.path[0])


## 3. Config


In [ ]:
MODEL_NAME = "Qwen/Qwen3-0.6B"
LAYER = 14
PROBE_LAYER = LAYER
COEF = 0.4                     # additive_normalized: 0.4 * ||h|| * v_hat
HYSTERESIS = 2
THRESHOLD = 0.5
MAX_EXAMPLES = int(os.environ.get("MAX_EXAMPLES", "20"))
MAX_NEW_TOKENS = int(os.environ.get("MAX_NEW_TOKENS", "256"))
RUN_TAG = time.strftime("reactive_%Y%m%d_%H%M%S")
RESULTS_DIR = Path("nb_results") / RUN_TAG
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"layer={LAYER} coef={COEF} max_examples={MAX_EXAMPLES} -> {RESULTS_DIR}")


## 4. Probe + steering vector


In [ ]:
# Load LR probe (weights + bias) and the steering vector at LAYER. Tries the
# train-derived CAA vector first (post-Issue-#5 fix), falls back to the
# val-derived legacy vector.
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/stvngo/Pivotal-Token-Representation-Learning/main/artifacts/cached3/sklearn/steering_configs"
ART_DIR = Path("artifacts/cached3/sklearn/steering_configs")

def _ensure(name: str) -> Path:
    local = ART_DIR / name if (ART_DIR / name).exists() else Path(name)
    if not local.exists():
        try:
            urllib.request.urlretrieve(f"{REPO_RAW}/{name}", name)
            local = Path(name)
        except Exception as e:
            print(f"  WARN: cannot fetch {name}: {e}")
    return local

w_path = _ensure(f"steering_layer{LAYER}_probe_weights.npy")
b_path = _ensure(f"steering_layer{LAYER}_probe_biases.npy")
v_train_path = _ensure(f"steering_layer{LAYER}_vector_train.npy")
v_val_path = _ensure(f"steering_layer{LAYER}_vector.npy")

probe_w = np.load(w_path).astype(np.float32).reshape(-1)
probe_b = float(np.load(b_path).reshape(-1)[0])
if v_train_path.exists():
    steering_vec = np.load(v_train_path).astype(np.float32).reshape(-1)
    VECTOR_SOURCE = "caa_train"
else:
    steering_vec = np.load(v_val_path).astype(np.float32).reshape(-1)
    VECTOR_SOURCE = "caa_val"
print(f"probe ||w||={np.linalg.norm(probe_w):.3f} b={probe_b:+.3f} "
      f"vec ||v||={np.linalg.norm(steering_vec):.3f} source={VECTOR_SOURCE}")


## 5. Load Qwen3-0.6B


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else (
    "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu"
)
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=DTYPE, trust_remote_code=True, low_cpu_mem_usage=True,
).to(DEVICE)
model.eval()
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
print(f"loaded {MODEL_NAME} on {DEVICE} dtype={DTYPE}; n_layers={len(model.model.layers)}")


## 5b. Convention check (Issue #2)


In [ ]:
# Convention check: hook on _get_decoder_layer(model, LAYER) modifies the same
# residual the LAYER-probe was trained on (Issue #2 in docs/issues.md).
def _convention_check(_model, _L):
    captured = {}
    def cap(_m, _i, out):
        h = out[0] if isinstance(out, tuple) else out
        captured["pre"] = h.detach()
        return out
    target = _get_decoder_layer(_model, _L)
    handle = target.register_forward_hook(cap)
    try:
        ids = torch.tensor([[1, 2, 3]], device=next(_model.parameters()).device)
        with torch.no_grad():
            outs = _model(input_ids=ids, output_hidden_states=True)
    finally:
        handle.remove()
    expected = outs.hidden_states[_L]
    return bool(torch.allclose(captured["pre"], expected, atol=1e-4, rtol=1e-3))

CONVENTION_CHECK_PASSED = _convention_check(model, LAYER)
assert CONVENTION_CHECK_PASSED, "Off-by-one regression; see docs/issues.md Issue #2"
print(f"[convention] hook == hidden_states[{LAYER}]: OK")


## 6. Data + parser + metrics


In [ ]:
from datasets import load_dataset

def extract_gsm8k_answer(text: str):
    m = re.search(r"####\s*([+-]?\d+(?:,\d{3})*(?:\.\d+)?)", text)
    if m:
        return m.group(1).replace(",", "").strip()
    nums = re.findall(r"[-+]?\d*\.?\d+", text)
    return nums[-1] if nums else None

def is_correct(p, g):
    if p is None or g is None: return False
    p = p.strip().replace(",", ""); g = g.strip().replace(",", "")
    try: return float(p) == float(g)
    except ValueError: return p == g

def compute_metrics(rows):
    n = len(rows)
    if n == 0: return {"accuracy": 0.0, "f1": 0.0, "parse_rate": 0.0, "correct": 0, "n": 0}
    c = sum(1 for r in rows if r["correct"])
    parsed = sum(1 for r in rows if r["predicted"] is not None)
    tp, fp, fn = c, parsed - c, n - c
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return {"accuracy": c / n, "f1": f1, "parse_rate": parsed / n, "correct": c, "n": n}

ds = load_dataset("openai/gsm8k", "main", split="test")
rng = np.random.default_rng(SEED)
idxs = sorted(rng.choice(len(ds), size=min(MAX_EXAMPLES, len(ds)), replace=False).tolist())
examples = [
    {"question": ds[i]["question"], "ground_truth": extract_gsm8k_answer(ds[i]["answer"])}
    for i in idxs
]
print(f"n_examples={len(examples)}")


## 7. Generation + run helper


In [ ]:
def generate_once(prompt: str) -> str:
    enc = tokenizer(prompt, return_tensors="pt", add_special_tokens=True).to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True, temperature=0.6, top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)


def run_arm(label: str, hook_ctx=None):
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    rows = []
    if hook_ctx is None:
        for i, ex in enumerate(examples):
            text = generate_once(f"Question: {ex['question']}\n\nLet's think step by step.\n\n")
            pred = extract_gsm8k_answer(text)
            ok = is_correct(pred, ex["ground_truth"])
            rows.append({"idx": i, "predicted": pred, "correct": ok,
                         "ground_truth": ex["ground_truth"],
                         "question": ex["question"], "full_output": text[-600:]})
    else:
        with hook_ctx as h:
            for i, ex in enumerate(examples):
                text = generate_once(f"Question: {ex['question']}\n\nLet's think step by step.\n\n")
                pred = extract_gsm8k_answer(text)
                ok = is_correct(pred, ex["ground_truth"])
                rows.append({"idx": i, "predicted": pred, "correct": ok,
                             "ground_truth": ex["ground_truth"],
                             "question": ex["question"], "full_output": text[-600:]})
            stats = h.stats.to_dict()
        return rows, compute_metrics(rows), stats
    return rows, compute_metrics(rows), {}


## 8. Three-arm sweep


In [ ]:
arms = {}

# ---- Arm 1: BASE ----
base_rows, base_met, _ = run_arm("base", None)
arms["base"] = {"results": base_rows, "metrics": base_met, "stats": {}}
print(f"base: acc={base_met['accuracy']:.3f} parse={base_met['parse_rate']:.2f}")

# ---- Arm 2: ALWAYS-ON additive_normalized at every position ----
always_hook = ReactiveSteeringHook(
    model, LAYER, probe_w, probe_b, steering_vec, COEF,
    mode="additive_normalized", threshold=-1e9, hysteresis=0,
    always_on_during_prefill=True,
)
always_rows, always_met, always_stats = run_arm("always_on", always_hook)
arms["always_on"] = {"results": always_rows, "metrics": always_met, "stats": always_stats}
print(f"always_on: acc={always_met['accuracy']:.3f} fire_rate={always_stats['fire_rate']:.2f}")

# ---- Arm 3: REACTIVE_DETECT (gate by binary probe) ----
detect_hook = ReactiveSteeringHook(
    model, LAYER, probe_w, probe_b, steering_vec, COEF,
    mode="additive_normalized", threshold=THRESHOLD, hysteresis=HYSTERESIS,
)
detect_rows, detect_met, detect_stats = run_arm("reactive_detect", detect_hook)
arms["reactive_detect"] = {"results": detect_rows, "metrics": detect_met, "stats": detect_stats}
print(f"reactive_detect: acc={detect_met['accuracy']:.3f} "
      f"fire_rate={detect_stats['fire_rate']:.2f} "
      f"holds={detect_stats['n_hysteresis_holds']}")

# ---- Arm 4: REACTIVE_RANDOM (matched fire rate, random positions) ----
matched_rate = max(0.05, min(0.95, detect_stats["fire_rate"] or 0.2))
n_steps = max(MAX_NEW_TOKENS * MAX_EXAMPLES + 32, 256)
pattern = build_random_fire_pattern(n_steps, matched_rate, seed=101)
rand_hook = ReactiveSteeringHook(
    model, LAYER, probe_w, probe_b, steering_vec, COEF,
    mode="additive_normalized", threshold=0.5, hysteresis=0,
    force_fire_pattern=pattern,
)
rand_rows, rand_met, rand_stats = run_arm("reactive_random", rand_hook)
arms["reactive_random"] = {"results": rand_rows, "metrics": rand_met, "stats": rand_stats}
print(f"reactive_random: acc={rand_met['accuracy']:.3f} "
      f"fire_rate={rand_stats['fire_rate']:.2f} "
      f"matched_to={matched_rate:.2f}")


## 9. Flip overlap + save state files


In [ ]:
import pandas as pd

# Flip overlap with base: how many predictions changed under each arm?
def flip_overlap(base, other):
    base_ok = {r["idx"]: r["correct"] for r in base}
    flips = sum(1 for r in other if base_ok.get(r["idx"]) != r["correct"])
    pos = sum(1 for r in other if r["correct"] and not base_ok.get(r["idx"], False))
    neg = sum(1 for r in other if not r["correct"] and base_ok.get(r["idx"], False))
    return {"total_flips": flips, "pos_flips": pos, "neg_flips": neg, "n": len(other)}

summary_rows = []
for name, arm in arms.items():
    if name == "base": continue
    summary_rows.append({
        "arm": name,
        "accuracy": arm["metrics"]["accuracy"],
        "parse_rate": arm["metrics"]["parse_rate"],
        "fire_rate": arm["stats"].get("fire_rate", float("nan")),
        "energy": arm["stats"].get("energy", float("nan")),
        "flips_vs_base": flip_overlap(arms["base"]["results"], arm["results"])["total_flips"],
    })

df = pd.DataFrame(summary_rows)
print(df)

# Save state files (label, metrics, plus provenance for the registry).
HOOK_TGT = LAYER - 1 if LAYER >= 1 else -1
def _state(label, metrics, mode, vector_source, extra=None):
    s = {
        "label": label, "metrics": metrics,
        "layer": LAYER, "hook_target_layer_idx": HOOK_TGT,
        "mode": mode, "vector_source": vector_source,
        "convention_check_passed": CONVENTION_CHECK_PASSED,
    }
    if extra: s.update(extra)
    return s

(RESULTS_DIR / "base_state.json").write_text(json.dumps(
    _state("base", arms["base"]["metrics"], "none", "n/a"), indent=2))
(RESULTS_DIR / "always_on_state.json").write_text(json.dumps(
    _state("always_on", arms["always_on"]["metrics"], "additive_normalized",
           VECTOR_SOURCE, extra={"arm": "always_on", "factor": 1.0 + COEF,
                                 "fire_rate": arms["always_on"]["stats"]["fire_rate"],
                                 "energy": arms["always_on"]["stats"]["energy"]}), indent=2))
(RESULTS_DIR / "reactive_detect_state.json").write_text(json.dumps(
    _state("reactive_detect", arms["reactive_detect"]["metrics"], "additive_normalized",
           f"probe_layer{LAYER}", extra={"arm": "reactive_detect", "factor": 1.0 + COEF,
                                         **arms["reactive_detect"]["stats"]}), indent=2))
(RESULTS_DIR / "reactive_random_state.json").write_text(json.dumps(
    _state("reactive_random", arms["reactive_random"]["metrics"], "additive_normalized",
           "random_pattern_seed101", extra={"arm": "reactive_random", "factor": 1.0 + COEF,
                                            "expected_fire_rate": matched_rate,
                                            **arms["reactive_random"]["stats"]}), indent=2))

(RESULTS_DIR / "summary.csv").write_text(df.to_csv(index=False))
print(f"saved -> {RESULTS_DIR}")


## 10. Bundle


In [ ]:
# Bundle outputs into a downloadable zip.
import shutil
zip_path = Path(f"nb_results_{RUN_TAG}.zip")
shutil.make_archive(zip_path.with_suffix(""), "zip", RESULTS_DIR.parent, RESULTS_DIR.name)
print(f"bundle -> {zip_path}")
try:
    from google.colab import files  # noqa: F401
    files.download(str(zip_path))
except Exception:
    pass
